# FoodFlow Multitask Training Sample

This notebook is a lightweight training and evaluation smoke test for the multitask FoodFlow GNN core package.

It does not run county-scale inference and does not generate prediction CSV files. It only loads the FAF multitask dataset, trains a small `MTLocalizedGCN`, and reports metrics such as ACC, AUC, R2, and CPF/CPC.

In [ ]:
from pathlib import Path
import sys

# This notebook is stored in code/, next to model.py, dataset.py, and train.py.
CODE_DIR = Path.cwd()
if not (CODE_DIR / "model.py").exists():
    # If launched from the repository root instead of code/, adjust paths.
    CODE_DIR = Path.cwd() / "code"

sys.path.insert(0, str(CODE_DIR.resolve()))
print(f"Using code directory: {CODE_DIR.resolve()}")

In [ ]:
import pandas as pd
import torch

from dataset import load_multitask_dataset, stratified_split_multitask
from model import MTLocalizedGCN
from train import (
    SCTG_NAMES,
    compute_pos_weights,
    eval_multitask_localized_gcn,
    training_loop,
)

torch.manual_seed(42)
print("Imports OK")

## 1. Load FAF multitask training data

The dataset stacks SCTG 01-07 targets into one multitask edge label matrix.

In [ ]:
dataset = load_multitask_dataset(n_pca=30, knn_k=5, edge_universe="all_pairs")

print("x:", tuple(dataset["x"].shape))
print("edge_index:", tuple(dataset["edge_index"].shape))
print("edge_attr:", tuple(dataset["edge_attr"].shape))
print("edge_y:", tuple(dataset["edge_y"].shape))
print("sparse_edge_index:", tuple(dataset["sparse_edge_index"].shape))

In [ ]:
train_data, val_data, test_data = stratified_split_multitask(
    dataset,
    train_ratio=0.70,
    val_ratio=0.15,
)

print("train edges:", train_data["edge_index"].shape[1])
print("val edges:", val_data["edge_index"].shape[1])
print("test edges:", test_data["edge_index"].shape[1])

## 2. Train a small MT-LocalizedGCN

This sample uses a tiny hidden size and only two epochs so the notebook runs quickly. For full training, use `python code/run_multitask.py`.

In [ ]:
node_dim = dataset["x"].shape[1]
edge_dim = dataset["edge_attr"].shape[1]
n_tasks = dataset["edge_y"].shape[1]

model = MTLocalizedGCN(
    node_dim=node_dim,
    edge_dim=edge_dim,
    hidden=16,
    n_tasks=n_tasks,
    dropout=0.2,
    sparse_edge_index=dataset["sparse_edge_index"],
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2)
pos_weights = compute_pos_weights(train_data["edge_y"])

history = training_loop(
    model=model,
    train_data=train_data,
    test_data=test_data,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=2,
    log_every=1,
    model_type="localized_gcn",
    pos_weights=pos_weights,
    val_data=val_data,
    selection_metric="mean_cpc",
)

## 3. Report ACC, CPF/CPC, and related metrics

`cpc` is the OD-level Common Part of Flows metric used here. It is the same overlap-style metric referred to as CPF/CPC in notes.

In [ ]:
with torch.no_grad():
    val_metrics = eval_multitask_localized_gcn(model, val_data)
    test_metrics = eval_multitask_localized_gcn(model, test_data)

def metrics_table(metrics):
    rows = []
    for code, m in metrics["per_task"].items():
        rows.append({
            "SCTG": code,
            "Category": SCTG_NAMES.get(code, f"SCTG {code}"),
            "ACC": m["accuracy"],
            "AUC": m["auc"],
            "R2": m["r2"],
            "CPF_CPC": m["cpc"],
            "Positive_R2": m["pos_r2"],
        })
    return pd.DataFrame(rows)

summary = pd.DataFrame([
    {
        "Split": "Validation",
        "Mean_R2": val_metrics["mean_r2"],
        "Mean_AUC": val_metrics["mean_auc"],
        "Mean_CPF_CPC": val_metrics["mean_cpc"],
        "Mean_Positive_R2": val_metrics["mean_pos_r2"],
    },
    {
        "Split": "Test",
        "Mean_R2": test_metrics["mean_r2"],
        "Mean_AUC": test_metrics["mean_auc"],
        "Mean_CPF_CPC": test_metrics["mean_cpc"],
        "Mean_Positive_R2": test_metrics["mean_pos_r2"],
    },
])

summary

In [ ]:
metrics_table(test_metrics)

## Full training command

For a normal training run outside this notebook:

```bash
python code/run_multitask.py
```